#  Energy Consumption Modeling for UAVs Based on Flight Data and Mission Parameters :
## 1. Data analysis and feature extraction

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

In [2]:
from utils import clean_flight_df, plot_flight_altitude_speed

### M100 Data

In [3]:
df = pd.read_csv(r"C:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\12683453\flights.csv", delimiter=",", encoding="utf-8", header=0)
cols = df.columns.tolist()

C:\Users\raman\AppData\Local\Temp\ipykernel_29780\2046151961.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\12683453\flights.csv", delimiter=",", encoding="utf-8", header=0)


### Data Cleaning

Sanity check

In [4]:
clean_df = clean_flight_df(df)

c:\Users\raman\Desktop\Kanchan_Material\01_AirTransportAndLogistics\Course_Material\Semester_03\Research Task\utils.py:222: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time_day'] = pd.to_datetime(df['time_day'], errors='coerce').dt.time


### Find different phase of flight using  altitude, horizontal and vertical speed. 

In [11]:
df = df.sort_values(["flight", "time"]).reset_index(drop=True)
df["altitude_measured"] = df["position_z"] - df.groupby("flight")["position_z"].transform("first")
df["horizontal_speed"] = np.sqrt(df["velocity_x"]**2 + df["velocity_y"]**2)
df['alt_diff'] = df.groupby('flight')['altitude_measured'].diff()
df['max_altitude_flight'] = df.groupby('flight')['altitude_measured'].transform('max')
df['min_altitude_flight'] = df.groupby('flight')['altitude_measured'].transform('min')

df["dt"] = df.groupby("flight")["time"].diff()

df["vz_from_alt_raw"] = df["alt_diff"] / df["dt"]

df["vz_from_alt"] = (
    df.groupby("flight")["vz_from_alt_raw"]
      .transform(lambda x: x.rolling(window=10, center=True, min_periods=1).mean())
)

### Velocity from altitude diff and velocity_z are not same e.g. in 120
So lets do a sanity check.

In [12]:
# Select flights to inspect
flights_to_plot = [120, 79, 279]   # add more flight numbers here if you like
sub = df[df["flight"].isin(flights_to_plot)].copy()

# Prepare data for Plotly (melt to long format for nice legends)
plot_df = sub.melt(
    id_vars=["flight", "time"],
    value_vars=["velocity_z", "vz_from_alt"],
    var_name="source",
    value_name="vz"
)

# Plot both velocities over time, faceted by Flight
fig = px.line(
    plot_df,
    x="time",
    y="vz",
    color="source",
    facet_row="flight",        # or facet_col="flight" if you prefer columns
    title="Reported vs Altitude-derived Vertical Velocity",
    labels={
        "time": "Time",
        "vz": "Vertical velocity (m/s)",
        "source": "Signal source"
    }
)

fig.update_layout(height=300*len(flights_to_plot))  # adjust height for number of flights
fig.show()

#### Definitions of various phases

In [25]:

# Thresholds
NEARLY_ZERO_HORIZONTAL_VEL   = 0.3   # m/s 
NEARLY_ZERO_VERTICAL_VEL     = 0.2   # m/s

CRUISE_MAX_VERTICAL_VEL    = 0.6   # m/s
CRUISE_MIN_VERTICAL_VEL    = -0.6  # m/s

TAXI_ALTITUDE_MAX      = 1.0   # m   # max altitude to be considered taxiing

# Start with a default phase
df['phase'] = 'other'

# --- Taxi: low altitude, nearly no motion ---
taxi_mask = (
    (df['horizontal_speed'].abs() < NEARLY_ZERO_HORIZONTAL_VEL) &
    (df['vz_from_alt'].abs()      < NEARLY_ZERO_VERTICAL_VEL) &
    (df['altitude_measured']      < TAXI_ALTITUDE_MAX)
)
df.loc[(df['phase'] == 'other') & taxi_mask, 'phase'] = 'taxi'


# --- Hover: nearly no motion ---
# 1. horizontal speed below hover max
# 2. vertical speed below hover max
hover_mask = (
    (df['horizontal_speed'].abs() < NEARLY_ZERO_HORIZONTAL_VEL) &
    (df['vz_from_alt'].abs()      < NEARLY_ZERO_VERTICAL_VEL) 
)
df.loc[(df['phase'] == 'other') & hover_mask, 'phase'] = 'hover'

# --- Cruise: nearly no vertical motion, high altitude ---
# 1. vertical speed below cruise max
# 2. altitude above 80% of max altitude in flight
cruise_mask = (
    (df['vz_from_alt'] < CRUISE_MAX_VERTICAL_VEL) &
    (df['vz_from_alt'] > CRUISE_MIN_VERTICAL_VEL) &
    (df['altitude_measured'] > 0.8 * df['max_altitude_flight'])
)
df.loc[(df['phase'] == 'other') & cruise_mask, 'phase'] = 'cruise'

# ---- CLIMB ----
# 1. vertical speed is greater than climb vertical speed minimum
climb_mask = (df['vz_from_alt'] > CRUISE_MAX_VERTICAL_VEL)
df.loc[(df['phase'] == 'other') & climb_mask, 'phase'] = 'climb'

# ---- DESCENT ----
# 1. vertical speed is less than negative of climb vertical speed minimum
descent_mask = (df['vz_from_alt'] < CRUISE_MIN_VERTICAL_VEL)
df.loc[(df['phase'] == 'other') & descent_mask, 'phase'] = 'descent'




### Visualize different phase of flight with Line graph

In [ ]:
flight_id = 79
d = df[df["flight"] == flight_id].copy()

# ------------------ Define colors for phases ------------------
phase_colors = {
    "climb":   "rgba(0,255,0,0.25)",      # light green
    "descent": "rgba(255,0,0,0.25)",      # light red
    "hover":   "rgba(255,165,0,0.25)",    # light orange
    "cruise":  "rgba(0,0,255,0.25)",      # light blue
    "taxi":    "rgba(255,255,0,0.25)",    # light yellow
    "other":   "rgba(150,150,150,0.15)"   # light gray
}

# ------------------ Build customdata for hover ------------------
customdata = np.stack((
    d["altitude_measured"],
    d["vz_from_alt"],
    d["horizontal_speed"],
    d["phase"]
), axis=-1)

fig = plot_flight_altitude_speed(
    d,
    customdata=customdata,
    phase_colors=phase_colors,
    flight_id=flight_id,
    show=True
)